![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [2]:
# Initialize QuantBook
qb = QuantBook()

# Example: Set Benchmark and add technology stocks
qb.SetBenchmark("SPY")
tech_stocks = [
    "AAPL", "MSFT"]

history_data = {}
options_data = {}
def black_scholes(S, K, T, r, sigma, option_type='call'):
    from scipy.stats import norm
    import numpy as np
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def calculate_iv(S, K, T, r, market_price, option_type='call'):
    from scipy.optimize import brentq
    def objective_function(sigma):
        return black_scholes(S, K, T, r, sigma, option_type) - market_price
    try:
        return brentq(objective_function, 1e-6, 5.0)  # Bounds for sigma
    except ValueError:
        return None

def find_comparable_option(options_data, underlying_price, moneyness_target=1.0):
    comparable_option = None
    min_diff = float('inf')

    for option in options_data:
        K = option['strike']
        moneyness = underlying_price / K
        if abs(moneyness - moneyness_target) < min_diff:
            min_diff = abs(moneyness - moneyness_target)
            comparable_option = option

    return comparable_option




In [27]:
for stock in tech_stocks:
    symbol = qb.AddEquity(stock, Resolution.Daily).Symbol
    qb.AddOption(symbol)  # Add options for the stock
    start_date = pd.Timestamp("2022-01-01")
    end_date = pd.Timestamp("2022-12-31")
    
    # Retrieve and process price history
    price_history = qb.History([symbol], start_date, end_date, Resolution.Daily).reset_index()
    price_history.set_index("time", inplace=True)
    price_history = price_history.add_prefix("underlying_")
    price_history["underlying_normalized_close"] = (price_history["underlying_close"] - price_history["underlying_close"].mean()) / price_history["underlying_close"].std()
    history_data[symbol.Value] = price_history  # Store the price data

    # Retrieve options history as a DataFrame
    options_history = qb.OptionHistory(symbol, start_date, end_date, Resolution.Daily).DataFrame
    options_history = options_history.reset_index()
    options_history = options_history.join(price_history[["underlying_close"]], on="time", how="left")
    options_history.rename(columns={"underlying_close": "underlying_price"}, inplace=True)
    options_history["days_to_expiry"] = (pd.to_datetime(options_history["expiry"]).dt.normalize() - pd.to_datetime(options_history["time"].dt.normalize())).dt.days

    # Add moneyness column for comparisons
    options_history["strike"] = pd.to_numeric(options_history["strike"], errors='coerce')
    options_history = options_history[options_history["strike"].notna()]
    options_history["moneyness"] = options_history["underlying_price"] / options_history["strike"].astype(float)

    options_data[symbol.Value] = []
    for _, row in options_history.iterrows():
        S = row["underlying_price"]
        K = row["strike"]
        T = row["days_to_expiry"] / 365.0
        r = 0.03  # Assume a constant risk-free rate
        market_price = row["close"]
        option_type = 'call' if row["type"] == 0 else 'put'


        iv = calculate_iv(S, K, T, r, market_price, option_type)
        options_data[symbol.Value].append({
            "strike": K,
            "expiry": row["expiry"],
            "implied_volatility": iv,
            "days_to_expiry": T * 365.0,
            "underlying_price": S,
            "moneyness": row["moneyness"]
        })

    print(f"Processed options data with underlying price for {stock}")

In [29]:
# Find comparable options for AAPL and MSFT
underlying_prices = {stock: history_data[stock].iloc[-1]['underlying_close'] for stock in tech_stocks}
aapl_option = find_comparable_option(options_data['AAPL'], underlying_prices['AAPL'])
msft_option = find_comparable_option(options_data['MSFT'], underlying_prices['MSFT'])

# Ensure comparable moneyness and same expiration date for both stocks
def find_most_comparable(options_a, options_b, target_moneyness=1.0):
    best_pair = None
    min_diff = float('inf')

    for option_a in options_a:
        for option_b in options_b:
            if option_a['expiry'] == option_b['expiry']:
                diff = abs(option_a['moneyness'] - target_moneyness) + abs(option_b['moneyness'] - target_moneyness)
                if diff < min_diff:
                    min_diff = diff
                    best_pair = (option_a, option_b)

    return best_pair

comparable_pair = find_most_comparable(options_data['AAPL'], options_data['MSFT'])

print("Comparable Options Pair:", comparable_pair)


In [21]:
# opt_history_obj = options_history.DataFrame
# print(opt_history_obj)

# slice_obj_items = slice_obj.quote_bars
# print(slice_obj_items)

strike_values = options_history["strike"].value_counts()
option_types = options_history['type'].value_counts()
option_determination = options_history[options_history['underlying_price'] >= 178.850834][:20]


print(options_history[:5])
print('-----------------------')


In [22]:
print(option_determination.head(10))

In [36]:
options_history["days_to_expiry"] = (pd.to_datetime(options_history["expiry"]) - pd.to_datetime(options_history["time"])).dt.days
# diropt = options_history["expiry"][:5] - datetime(options_history["time"][:5])
# print(diropt)

In [38]:
print(options_history["days_to_expiry"][:20])

In [ ]:
# Find comparable options for AAPL and MSFT
underlying_prices = {stock: history_data[stock].iloc[-1]['close'] for stock in tech_stocks}
aapl_option = find_comparable_option(options_data['AAPL'], underlying_prices['AAPL'])
msft_option = find_comparable_option(options_data['MSFT'], underlying_prices['MSFT'])

print("Comparable AAPL Option:", aapl_option)
print("Comparable MSFT Option:", msft_option)

## NEW INVESTIGATION FOR QC Objects


In [ ]:
# Initialize QuantBook
qb = QuantBook()
qb_dir = dir(qb) ## methods and classes callable from QuantBook class

print("QuantBook Methods: \n", qb_dir)


In [3]:
QC = QCAlgorithmFramework
qc_dir = dir(QC)
print("QC Methods: \n", qc_dir)

Define Universe of Stocks to Analyze

In [38]:
universe_symbols = ["AAPL", "MSFT", "NVDA", "GOOG"]
qb.SetBenchmark("SPY")
universe_resolution = Resolution.Daily
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2024-12-31")

Add Stocks and Options to the QB Object.

In [ ]:
for stock in universe_symbols:
    symbol = qb.AddEquity(stock, universe_resolution).Symbol # Adds Equity and Symbol
    qb.AddOption(symbol) # Adds Option
    print(f"Equity and Option Added: {symbol}")

    # Adds Stock History
    stock_history = qb.History([symbol], start_date, end_date, universe_resolution).reset_index()
    stock_history.set_index("time", inplace=True)
    stock_history = stock_history.add_prefix("underlying_")

    # Adds Option History
    option_history = qb.OptionHistory(symbol, start_date, end_date, universe_resolution).DataFrame
    option_history = option_history.reset_index()
    option_history = option_history.join(stock_history[["underlying_close"]], on="time", how="left")
    option_history.rename(columns={"underlying_close": "underlying_price"}, inplace=True)
    option_history["days_to_expiry"] = (pd.to_datetime(option_history["expiry"]).dt.normalize() - pd.to_datetime(option_history["time"].dt.normalize())).dt.days

Clean the dataframe dropping all error-prone records

In [40]:
# Ensure numeric types for calculations
option_history['underlying_price'] = pd.to_numeric(option_history['underlying_price'], errors='coerce')
option_history['strike'] = pd.to_numeric(option_history['strike'], errors='coerce')

Add Additional calculated columns

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq

# Define Black-Scholes model function
def black_scholes(S, K, T, r, sigma, option_type='call'):
    try:
        if sigma <= 0 or T <= 0:
            return np.nan
        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if option_type == 'call':
            return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    except ZeroDivisionError:
        return np.nan

# Define Implied Volatility calculation
def calculate_iv(S, K, T, r, market_price, option_type='call'):
    def objective_function(sigma):
        return black_scholes(S, K, T, r, sigma, option_type) - market_price
    try:
        return brentq(objective_function, 1e-6, 5.0)  # Bounds for sigma
    except ValueError:
        return None


options_history = pd.DataFrame(option_history)

# Add derived columns
options_history['T'] = options_history['days_to_expiry'] / 365.0  # Convert days to years
options_history['risk_free_rate'] = 0.03  # Assume constant risk-free rate

# Calculate Black-Scholes price and Implied Volatility for each row
def process_row(row):
    S = row['underlying_price']
    K = row['strike']
    T = row['T']
    r = row['risk_free_rate']
    market_price = row['close']
    option_type = row['type']
    sigma = 0.2  # Initial guess for implied volatility

    bs_price = black_scholes(S, K, T, r, sigma, option_type)
    iv = calculate_iv(S, K, T, r, market_price, option_type)
    return pd.Series({'bs_price': bs_price, 'implied_volatility': iv})

# Apply to DataFrame
options_history[['bs_price', 'implied_volatility']] = options_history.apply(process_row, axis=1)

# Add theoretical_diff column
options_history['theoretical_diff'] = options_history['bs_price'] - options_history['close']
options_history['valuation'] = np.where(options_history['theoretical_diff'] > 0, 'undervalued', 'overvalued')

Display the dataset inside QC Notebook

In [7]:
 # Display DataFrame with filterable headers
from IPython.core.display import HTML
HTML("""
    <script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
    <script src="https://cdn.datatables.net/1.11.3/js/jquery.dataTables.min.js"></script>
    <link rel="stylesheet" href="https://cdn.datatables.net/1.11.3/css/jquery.dataTables.min.css">
    <script>
        $(document).ready(function() {
            $('table').DataTable();
        });
    </script>
""" + options_history.head(10).to_html())



In [8]:
distinct_values = options_history[['symbol', 'expiry']].drop_duplicates()
print(distinct_values)


Graph the columns from the dataframe

In [15]:
# Visualization block: Graphing columns over time for a specific symbol
import matplotlib.pyplot as plt
import seaborn as sns

def plot_symbol_data(symbol, y_columns):
    """
    Plots the specified columns against time for a specific symbol.

    Parameters:
        symbol (str): The symbol of the option contract to filter.
        y_columns (list): List of columns to plot on the Y-axis.
    """
    filtered_data = options_history[options_history['symbol'] == symbol]
    if filtered_data.empty:
        print(f"No data available for symbol: {symbol}")
        return

    plt.figure(figsize=(12, 6))
    for col in y_columns:
        if col in filtered_data.columns:
            plt.plot(filtered_data['time'], filtered_data[col], label=col)
        else:
            print(f"Column {col} not found in data.")

    plt.title(f"Data Visualization for Symbol: {symbol}")
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend()
    plt.grid()
    plt.show()

# Example usage:
plot_symbol_data('GOOCV YP8G6IM1GLHI|GOOCV VP83T1ZUHROL', ['bs_price', 'askclose', 'implied_volatility', 'theoretical_diff'])

In [13]:
def plot_all_contracts(symbol_prefix, y_columns):
    """
    Plots the specified columns against time for all contracts starting with a given symbol prefix.

    Parameters:
        symbol_prefix (str): The base symbol prefix (e.g., "GOOCV") of the option contracts to filter.
        y_columns (list): List of columns to plot on the Y-axis.
    """
    # Verify required columns exist
    for col in ['symbol', 'time'] + y_columns:
        if col not in options_history.columns:
            print(f"Column '{col}' is missing in options_history.")
            return

    # Filter data for the given symbol prefix
    filtered_data = options_history[options_history['symbol'].str.startswith(symbol_prefix)]
    print(f"Filtered data for symbol prefix '{symbol_prefix}' with {len(filtered_data)} rows.")

    if filtered_data.empty:
        print(f"No data available for symbol prefix: {symbol_prefix}")
        return

    contracts = filtered_data['symbol'].unique()
    print(f"Found {len(contracts)} distinct contracts for symbol prefix '{symbol_prefix}'.")

    for contract in contracts:
        contract_data = filtered_data[filtered_data['symbol'] == contract]
        print(f"Plotting data for contract '{contract}' with {len(contract_data)} rows.")

        # Drop NaN values in the columns to be plotted
        contract_data = contract_data.dropna(subset=['time'] + y_columns)

        plt.figure(figsize=(12, 6))
        for col in y_columns:
            if col in contract_data.columns:
                plt.plot(contract_data['time'], contract_data[col], label=col)
            else:
                print(f"Column {col} not found in data for contract: {contract}")

        plt.title(f"Data Visualization for Contract: {contract}")
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.legend()
        plt.grid()
        plt.show()

# Example usage:
plot_all_contracts('GOOCV', ['askclose', 'theoretical_diff'])


In [13]:
print("Columns in options_history:", options_history.columns)


In [11]:
print(f"Options history before dropping NaN rows: {len(options_history)} rows.")
options_history = options_history.dropna(subset=['symbol', 'time', 'askclose', 'theoretical_diff'])
print(f"Options history after dropping NaN rows: {len(options_history)} rows.")


In [14]:
filtered_data = options_history[options_history['symbol'].str.startswith('GOOCV')]
print(filtered_data.head())
print(f"Filtered data contains {len(filtered_data)} rows.")


In [16]:
print(options_history['symbol'].unique())

In [17]:
# Create a copy to avoid SettingWithCopyWarning
options_history = options_history.copy()

# Convert `symbol` column to strings
options_history['symbol'] = options_history['symbol'].apply(str)

print(f"Converted 'symbol' column to strings. Sample values: {options_history['symbol'].head()}")



In [ ]:
filtered_data = options_history[options_history['symbol'].str.startswith('GOOCV')]
print(f"Filtered data contains {len(filtered_data)} rows.")

if not filtered_data.empty:
    plot_all_contracts('GOOCV', ['askclose', 'theoretical_diff'])
else:
    print("No data available for symbol prefix 'GOOCV'.")
